In [36]:
%pip install matplotlib
import matplotlib.pyplot as plt
import numpy as np
%pip install pandas
import pandas as pd
import torch
%pip install scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

print("Library Versions:")
print('numpy:',np.__version__)
print('pandas:',pd.__version__)
print('torch:',torch.__version__)


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Library Versions:
numpy: 1.24.3
pandas: 2.0.3
torch: 2.4.0


In [37]:

n_epochs = 200
verbose_option = True

# Regression for Naval Plant Maintenance

Load regression dataset

In [38]:
npm = pd.read_csv("C:/Users/edgar/Downloads/navalplantmaintenance.csv",header=None)
npm

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
0,1.14,3.0,290.0,1350.0,6680.0,7.58,7.58,464.0,288.0,551.0,1.10,0.998,5.95,1.02,7.14,0.082,0.95,0.975
1,2.09,6.0,6960.0,1380.0,6830.0,28.20,28.20,635.0,288.0,582.0,1.33,0.998,7.28,1.02,10.70,0.287,0.95,0.975
2,3.14,9.0,8380.0,1390.0,7110.0,60.40,60.40,606.0,288.0,588.0,1.39,0.998,7.57,1.02,13.10,0.259,0.95,0.975
3,4.16,12.0,14700.0,1550.0,7790.0,114.00,114.00,661.0,288.0,614.0,1.66,0.998,9.01,1.02,18.10,0.358,0.95,0.975
4,5.14,15.0,21600.0,1920.0,8490.0,175.00,175.00,731.0,288.0,646.0,2.08,0.998,11.20,1.03,26.40,0.522,0.95,0.975
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11929,5.14,15.0,21600.0,1920.0,8470.0,175.00,175.00,682.0,288.0,629.0,2.09,0.998,11.00,1.03,23.80,0.471,1.00,1.000
11930,6.18,18.0,29800.0,2310.0,8800.0,246.00,246.00,747.0,288.0,659.0,2.51,0.998,13.10,1.03,32.70,0.647,1.00,1.000
11931,7.15,21.0,39000.0,2680.0,9120.0,332.00,332.00,796.0,288.0,680.0,2.98,0.998,15.40,1.04,42.10,0.834,1.00,1.000
11932,8.21,24.0,51000.0,3090.0,9300.0,438.00,438.00,893.0,288.0,722.0,3.59,0.998,18.30,1.04,58.10,1.150,1.00,1.000


Split dataset into training and test datasets

In [39]:
npm_train, npm_test = train_test_split(npm,test_size=0.25,random_state=42)

1. Create the training and test input matrices using the first 16 columns and the training and test target matrices using the 18th columns.

In [40]:
npm_train_np = npm_train.to_numpy()
npm_test_np = npm_test.to_numpy()
#TODO: Create training input matrix
x_train_np = npm_train_np[:,0:16]
#TODO: Create test input matrix
x_test_np = npm_test_np[:,0:16]
#TODO: Create training target matrix
y_train_np = npm_train_np[:,0:18]
#TODO: Create test target matrix
y_test_np = npm_test_np[:,0:18]

2. Z-score the input and output data using the training dataset statistics

In [41]:
#TODO: Compute the mean vector for the input features across examples
x_mu = np.mean(x_train_np,axis=0)
#TODO: Compute the standard deviation vector for the input features across examples
x_sigma = np.std(x_train_np,axis=0)
#TODO: Compute the mean for the targets across examples
y_mu = np.mean(y_train_np,axis=0)
#TODO: Compute the standard deviation for the targets across examples
y_sigma = np.std(y_train_np,axis=0)

In [42]:
def scale(x,x_mu,x_sigma):
  x_sigma += 1e-16 #This is to deal with constant or near-constant columns
  #TODO: Return the z-scored matrix
  return (x-x_mu)/x_sigma

In [43]:
def unscale(x,x_mu,x_sigma):
  x_sigma += 1e-16 #This is to deal with constant or near-constant columns
  #TODO: Return the un-z-scored matrix
  return x*x_sigma + x_mu

In [44]:
x_train_np_z = scale(x_train_np,x_mu,x_sigma)
y_train_np_z = scale(y_train_np,y_mu,y_sigma)
x_test_np_z = scale(x_test_np,x_mu,x_sigma)
y_test_np_z = scale(y_test_np,y_mu,y_sigma)

3. Using PyTorch, perform Maximum Likelihood Estimation (MLE) for a linear Gaussian
prediction model with homoscedastic uncertainty for the regression dataset.

In [45]:
def nlls(y, mu, std):
    return torch.log(std) + torch.square(y - mu)/(2.0*torch.square(std))

In [46]:

class linear(torch.nn.Module):
    def __init__(self, inputSize, outputSize):
        super(linear, self).__init__()
        #TODO: Call a linear layer constructor
        self.linear = torch.nn.Linear(inputSize,outputSize)
        #TODO: Call a parameter constructor
        self.s = torch.nn.Parameter(torch.zeros(outputSize))
    
    def forward(self, x):
        #TODO: Call the linear layer on x
        out = self.linear(x)
        return out, torch.nn.functional.softplus(self.s)

In [72]:
x_train_t_z = torch.FloatTensor(x_train_np_z)
y_train_t_z = torch.FloatTensor(y_train_np_z)

model = linear(x_train_np_z.shape[1],1)
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.05)
for i in range(n_epochs):
    #TODO: Call the model on the training data
    mu,s = model(x_train_t_z)
    #TODO: Define the loss using the nlls function
    loss = nlls(y_train_t_z,mu,s).mean()
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()  
    if verbose_option: print(loss)

tensor(0.6213, grad_fn=<MeanBackward0>)
tensor(0.0119, grad_fn=<MeanBackward0>)
tensor(0.1764, grad_fn=<MeanBackward0>)
tensor(0.3019, grad_fn=<MeanBackward0>)
tensor(0.2003, grad_fn=<MeanBackward0>)
tensor(0.0634, grad_fn=<MeanBackward0>)
tensor(0.0165, grad_fn=<MeanBackward0>)
tensor(0.0654, grad_fn=<MeanBackward0>)
tensor(0.1323, grad_fn=<MeanBackward0>)
tensor(0.1412, grad_fn=<MeanBackward0>)
tensor(0.0866, grad_fn=<MeanBackward0>)
tensor(0.0166, grad_fn=<MeanBackward0>)
tensor(-0.0182, grad_fn=<MeanBackward0>)
tensor(-0.0028, grad_fn=<MeanBackward0>)
tensor(0.0324, grad_fn=<MeanBackward0>)
tensor(0.0419, grad_fn=<MeanBackward0>)
tensor(0.0112, grad_fn=<MeanBackward0>)
tensor(-0.0324, grad_fn=<MeanBackward0>)
tensor(-0.0510, grad_fn=<MeanBackward0>)
tensor(-0.0355, grad_fn=<MeanBackward0>)
tensor(-0.0157, grad_fn=<MeanBackward0>)
tensor(-0.0241, grad_fn=<MeanBackward0>)
tensor(-0.0541, grad_fn=<MeanBackward0>)
tensor(-0.0707, grad_fn=<MeanBackward0>)
tensor(-0.0571, grad_fn=<MeanBa

4. For the trained linear regression model, get the mean and the standard deviation for the first test example.



In [48]:
x_test_t_z = torch.FloatTensor(x_test_np_z)
y_test_t_z = torch.FloatTensor(y_test_np_z)
#TODO: Call the model on the test data
y_test_mu_t_z, y_test_sigma_t_z = model(x_test_t_z)
#TODO: Calculate the un-z-scored predicted mean
y_test_pred_mu_t = unscale(y_test_mu_t_z.detach(), y_mu, y_sigma)
y_test_pred_mu = y_test_pred_mu_t.detach().numpy()
print('Model mean:', y_test_pred_mu)
#TODO: Calculate the un-z-scored predicted standard deviation
y_test_pred_sigma_t = y_test_sigma_t_z.detach()*y_sigma
print('Model standard deviation:', y_test_pred_sigma_t.detach().numpy())

Model mean: [[3.38530642e+00 9.74452747e+00 1.22040571e+04 ... 3.17903100e-01
  9.65034967e-01 9.82472502e-01]
 [9.10882291e+00 2.66206805e+01 6.05184323e+04 ... 1.42419359e+00
  9.97161809e-01 9.98790922e-01]
 [6.48312862e+00 1.88786536e+01 3.83539501e+04 ... 9.16676815e-01
  9.82423446e-01 9.91304758e-01]
 ...
 [9.03167918e+00 2.63932172e+01 5.98672328e+04 ... 1.40928259e+00
  9.96728791e-01 9.98570976e-01]
 [2.80126023e+00 8.02243007e+00 7.27390149e+03 ... 2.05013630e-01
  9.61756640e-01 9.80807318e-01]
 [9.11289064e+00 2.66326744e+01 6.05527696e+04 ... 1.42497983e+00
  9.97184642e-01 9.98802520e-01]]
Model standard deviation: [1.47357665e+00 4.34493463e+00 1.24390199e+04 4.34375027e+02
 6.12178122e+02 1.12538665e+02 1.12538665e+02 9.75234322e+01
 1.68278182e-16 4.07898090e+01 6.08772698e-01 2.30553619e-16
 2.99456532e+00 5.92205771e-03 1.44900387e+01 2.84825568e-01
 8.27137729e-03 4.20134068e-03]


5. Compute the Mean Squared Error (MSE) for the trained linear homoscedastic regression model for the test data

In [49]:
#TODO: Compute MSE for the model on the test data
mse_test = mean_squared_error(y_test_np, y_test_pred_mu)
print('MSE:', mse_test)

MSE: 1624261.6834794027


6. Using PyTorch, perform Maximum Likelihood Estimation (MLE) for a non-linear Gaussian
prediction model with homoscedastic uncertainty for the regression dataset.

In [50]:
class nn(torch.nn.Module):
    def __init__(self, inputSize, hiddenSize, outputSize):
        super(nn, self).__init__()
        #TODO: Call a linear layer constructor
        self.layer1 = torch.nn.Linear(inputSize,hiddenSize)
        #TODO: Call a linear layer constructor
        self.layer2 = torch.nn.Linear(hiddenSize,hiddenSize)
        #TODO: Call a linear layer constructor
        self.linear_mu = torch.nn.Linear(hiddenSize,outputSize)
        #TODO: Call a parameter constructor
        self.s = torch.nn.Parameter(torch.zeros(outputSize))

    def forward(self, x):
        #TODO: Call layer 1 on x using an ReLU activation function
        h1 = torch.nn.functional.relu(self.layer1(x))
        #TODO: Call layer 2 on h1 using an ReLU activation function
        h2 = torch.nn.functional.relu(self.layer2(h1))
        #TODO: Call linear_mu on h2
        mu = self.linear_mu(h2)
        linear_mu = self.linear_mu(h2)
        sigma = torch.nn.functional.softplus(self.s)
        return mu, sigma

In [51]:
model = nn(x_train_np_z.shape[1],50, 1)
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.05)
for i in range(n_epochs):
    #TODO: Call the model on the training data
    mu, s = model(x_train_t_z)
    #TODO: Define the loss using the nlls function
    loss = nlls(y_train_t_z,mu,s).mean()
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()  
    if verbose_option: print(loss)

tensor(0.5953, grad_fn=<MeanBackward0>)


tensor(0.8217, grad_fn=<MeanBackward0>)
tensor(1.0318, grad_fn=<MeanBackward0>)
tensor(0.3550, grad_fn=<MeanBackward0>)
tensor(0.1086, grad_fn=<MeanBackward0>)
tensor(0.1562, grad_fn=<MeanBackward0>)
tensor(0.0791, grad_fn=<MeanBackward0>)
tensor(0.2043, grad_fn=<MeanBackward0>)
tensor(0.0678, grad_fn=<MeanBackward0>)
tensor(0.2212, grad_fn=<MeanBackward0>)
tensor(0.0631, grad_fn=<MeanBackward0>)
tensor(0.1711, grad_fn=<MeanBackward0>)
tensor(0.0697, grad_fn=<MeanBackward0>)
tensor(0.1446, grad_fn=<MeanBackward0>)
tensor(0.0536, grad_fn=<MeanBackward0>)
tensor(0.1194, grad_fn=<MeanBackward0>)
tensor(0.0511, grad_fn=<MeanBackward0>)
tensor(0.0613, grad_fn=<MeanBackward0>)
tensor(0.0558, grad_fn=<MeanBackward0>)
tensor(0.0241, grad_fn=<MeanBackward0>)
tensor(0.0398, grad_fn=<MeanBackward0>)
tensor(0.0051, grad_fn=<MeanBackward0>)
tensor(0.0138, grad_fn=<MeanBackward0>)
tensor(-0.0062, grad_fn=<MeanBackward0>)
tensor(-0.0184, grad_fn=<MeanBackward0>)
tensor(-0.0111, grad_fn=<MeanBackward0

7. Compute the Mean Squared Error (MSE) for the trained non-linear homoscedastic regression model for the test data

In [52]:
x_test_t_z = torch.FloatTensor(x_test_np_z)
y_test_t_z = torch.FloatTensor(y_test_np_z)
y_test_mu_t_z, y_test_sigma_t_z = model(x_test_t_z)
#TODO: Calculate the un-z-scored predicted mean
y_test_pred_mu_t = unscale(y_test_mu_t_z.detach(), y_mu, y_sigma)
y_test_pred_mu = y_test_pred_mu_t.detach().numpy()
print('Model mean:', y_test_pred_mu)
#TODO: Calculate the un-z-scored predeicted standard deviation
y_test_pred_mu_t = unscale(y_test_mu_t_z.detach(), y_mu, y_sigma)
print('Model standard deviation:', y_test_pred_sigma_t.detach().numpy())

Model mean: [[3.49793967e+00 1.00766338e+01 1.31548371e+04 ... 3.39673823e-01
  9.65667192e-01 9.82793633e-01]
 [9.03447169e+00 2.64014511e+01 5.98908055e+04 ... 1.40982235e+00
  9.96744466e-01 9.98578938e-01]
 [6.52657548e+00 1.90067594e+01 3.87207016e+04 ... 9.25074598e-01
  9.82667318e-01 9.91428630e-01]
 ...
 [8.97674621e+00 2.62312439e+01 5.94035228e+04 ... 1.39866467e+00
  9.96420445e-01 9.98414356e-01]
 [2.74385120e+00 7.85315589e+00 6.78929008e+03 ... 1.93917119e-01
  9.61434395e-01 9.80643638e-01]
 [9.03617094e+00 2.64064615e+01 5.99051495e+04 ... 1.41015079e+00
  9.96754004e-01 9.98583783e-01]]
Model standard deviation: [1.47357665e+00 4.34493463e+00 1.24390199e+04 4.34375027e+02
 6.12178122e+02 1.12538665e+02 1.12538665e+02 9.75234322e+01
 1.68278182e-16 4.07898090e+01 6.08772698e-01 2.30553619e-16
 2.99456532e+00 5.92205771e-03 1.44900387e+01 2.84825568e-01
 8.27137729e-03 4.20134068e-03]


In [53]:
#TODO: Compute MSE for the model on the test data
mse_test = mean_squared_error(y_test_np, y_test_pred_mu) 

print('MSE:', mse_test)

MSE: 1852755.5080284912


8. Using PyTorch, perform Maximum Likelihood Estimation (MLE) for a non-linear Gaussian
prediction model with heteroscedastic uncertainty for the regression dataset.

In [54]:
model = nn(x_train_np_z.shape[1],50, 1)
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.05)
for i in range(n_epochs):
    #TODO: Call the model on the training data
    mu, s  = model(x_train_t_z)
    #TODO: Define the loss using the nlls function
    loss = nlls(y_train_t_z,mu,s).mean()
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()  
    if verbose_option: print(loss)

tensor(0.6669, grad_fn=<MeanBackward0>)
tensor(0.1934, grad_fn=<MeanBackward0>)
tensor(4.5311, grad_fn=<MeanBackward0>)
tensor(0.1728, grad_fn=<MeanBackward0>)
tensor(0.8437, grad_fn=<MeanBackward0>)
tensor(0.2253, grad_fn=<MeanBackward0>)
tensor(0.1375, grad_fn=<MeanBackward0>)
tensor(0.1821, grad_fn=<MeanBackward0>)
tensor(0.0554, grad_fn=<MeanBackward0>)
tensor(0.1466, grad_fn=<MeanBackward0>)
tensor(0.0877, grad_fn=<MeanBackward0>)
tensor(0.1021, grad_fn=<MeanBackward0>)
tensor(0.0933, grad_fn=<MeanBackward0>)
tensor(0.0928, grad_fn=<MeanBackward0>)
tensor(0.0832, grad_fn=<MeanBackward0>)
tensor(0.0952, grad_fn=<MeanBackward0>)
tensor(0.0909, grad_fn=<MeanBackward0>)
tensor(0.0667, grad_fn=<MeanBackward0>)
tensor(0.0997, grad_fn=<MeanBackward0>)
tensor(0.0611, grad_fn=<MeanBackward0>)
tensor(0.0793, grad_fn=<MeanBackward0>)
tensor(0.0676, grad_fn=<MeanBackward0>)
tensor(0.0687, grad_fn=<MeanBackward0>)
tensor(0.0503, grad_fn=<MeanBackward0>)
tensor(0.0717, grad_fn=<MeanBackward0>)


9. Compute the Mean Squared Error (MSE) for the trained non-linear heteroscedastic regression model for the test data

In [55]:
x_test_t_z = torch.FloatTensor(x_test_np_z)
y_test_t_z = torch.FloatTensor(y_test_np_z)
y_test_mu_t_z, y_test_sigma_t_z = model(x_test_t_z)
#TODO: Calculate the un-z-scored predicted mean
y_test_pred_mu_t = unscale(y_test_mu_t_z.detach(), y_mu, y_sigma)
y_test_pred_mu = y_test_pred_mu_t.detach().numpy()
print('Model mean:', y_test_pred_mu)
#TODO: Calculate the un-z-scored predeicted standard deviation
y_test_pred_sigma_t = y_test_sigma_t_z.detach()*y_sigma
print('Model standard deviation:', y_test_pred_sigma_t.detach().numpy())

Model mean: [[3.43300570e+00 9.88517183e+00 1.26067048e+04 ... 3.27122827e-01
  9.65302709e-01 9.82608498e-01]
 [9.17191702e+00 2.68067175e+01 6.10510337e+04 ... 1.43638896e+00
  9.97515964e-01 9.98970811e-01]
 [6.59479488e+00 1.92079087e+01 3.92965675e+04 ... 9.38260631e-01
  9.83050243e-01 9.91623132e-01]
 ...
 [9.11867452e+00 2.66497286e+01 6.06015936e+04 ... 1.42609779e+00
  9.97217107e-01 9.98819010e-01]
 [2.80177993e+00 8.02396244e+00 7.27828848e+03 ... 2.05114082e-01
  9.61759557e-01 9.80808799e-01]
 [9.17676454e+00 2.68210107e+01 6.10919534e+04 ... 1.43732593e+00
  9.97543174e-01 9.98984632e-01]]
Model standard deviation: [1.47462733e+00 4.34803262e+00 1.24478890e+04 4.34684741e+02
 6.12614612e+02 1.12618906e+02 1.12618906e+02 9.75929674e+01
 3.36796331e-16 4.08188926e+01 6.09206759e-01 3.99116172e-16
 2.99670047e+00 5.92628020e-03 1.45003703e+01 2.85028652e-01
 8.27727487e-03 4.20433628e-03]


In [56]:
#TODO: Compute MSE for the model on the test data
mse_test = mean_squared_error(y_test_np, y_test_pred_mu)
print('MSE:', mse_test)

MSE: 1551272.0438427492


# Classification for Ship Detection


Load Ship Detection Dataset

In [64]:
import torch 
from torch.utils.data import Dataset, DataLoader
#%pip install torchvision
from torchvision.io import read_image
from torch.utils.data import random_split
from torchvision.transforms.functional import resize
from sklearn import preprocessing
import numpy as np
from pathlib import Path
%pip install torchmetrics
import torchmetrics

ROOT_PATH = "C:/Users/edgar/Downloads/shipsnet/shipsnet"
LR = 1e-4
IMG_SIZE = [80]

tensor_size = IMG_SIZE[0]**2 * 3

def max_scaling(image):
    image = image / 255.0
    image = torch.Tensor(image)
    image = resize(image, size=IMG_SIZE)
    return image

def normalize_img(image):
    means = torch.Tensor([[[105.0385]],[[108.1886]],[[ 94.9558]]])
    stds = torch.Tensor([[[48.4294]],[[40.0104]],[[38.6445]]])
    image = torch.Tensor(image)
    image = resize(image, size=IMG_SIZE)
    image = image - means
    image = image / stds
    return image

#https://pytorch.org/tutorials/beginner/basics/data_tutorial.html
class ShipDataset(Dataset):
    def __init__(self, root_path, transform = None):
        self.root_path = Path(root_path)
        self.files = list(self.root_path.rglob("*/*"))
        self.classes = list(set([(entry.parts[-1]) for entry in self.root_path.rglob("*") if Path(entry).is_dir()]))
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        image = read_image(str(self.files[idx]))
        label = int(self.files[idx].parts[-2])
        if self.transform:
            image = self.transform(image)
        return image, label
    

full_dataset  = ShipDataset(ROOT_PATH, transform = normalize_img,)
n_classes = len(full_dataset.classes)
train_dataset, test_dataset = random_split(full_dataset, [0.8, 0.2])

train_dataloader = DataLoader(train_dataset, batch_size=len(train_dataset))
test_dataloader = DataLoader(test_dataset, batch_size=len(test_dataset))

criterion = torch.nn.BCELoss(reduce='mean')
accuracy = torchmetrics.classification.BinaryAccuracy()


Note: you may need to restart the kernel to use updated packages.


c:\Users\edgar\anaconda3\envs\pytorch_env\lib\site-packages\torch\nn\_reduction.py:42: UserWarning: size_average and reduce args will be deprecated, please use reduction='mean' instead.
  warnings.warn(warning.format(ret))


10. Using PyTorch, perform Maximum Likelihood Estimation (MLE) for a linear Bernoulli
prediction model (i.e., logistic regression) for the binary classification dataset.

In [65]:
class logistic(torch.nn.Module):
    def __init__(self, inputSize):
        super(logistic, self).__init__()
        #TODO: Call a linear layer constructor
        self.linear = torch.nn.Linear(inputSize,1)

    def forward(self, x):
        x = torch.flatten(x, start_dim=1)
        #TODO: Call linear on x using a sigmoid activation function
        return torch.sigmoid(self.linear(x))

In [67]:
model = logistic(tensor_size)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(n_epochs):
    for data, label in train_dataloader:
        optimizer.zero_grad()
        #TODO: Call the model on the training data
        p = model(data)
        loss = criterion(p.squeeze(),label*1.0) 
        acc = accuracy(p.squeeze(), label)
        loss.backward()
        optimizer.step()
        if verbose_option: print(epoch, loss, acc)

0 tensor(0.7214, grad_fn=<BinaryCrossEntropyBackward0>) tensor(0.4762)
1 tensor(4.1265, grad_fn=<BinaryCrossEntropyBackward0>) tensor(0.5391)
2 tensor(0.8947, grad_fn=<BinaryCrossEntropyBackward0>) tensor(0.5725)
3 tensor(1.9611, grad_fn=<BinaryCrossEntropyBackward0>) tensor(0.5387)
4 tensor(0.8005, grad_fn=<BinaryCrossEntropyBackward0>) tensor(0.6094)
5 tensor(1.4457, grad_fn=<BinaryCrossEntropyBackward0>) tensor(0.5931)
6 tensor(1.5627, grad_fn=<BinaryCrossEntropyBackward0>) tensor(0.5913)
7 tensor(0.9041, grad_fn=<BinaryCrossEntropyBackward0>) tensor(0.6263)
8 tensor(1.0843, grad_fn=<BinaryCrossEntropyBackward0>) tensor(0.6019)
9 tensor(1.3184, grad_fn=<BinaryCrossEntropyBackward0>) tensor(0.5800)
10 tensor(0.7845, grad_fn=<BinaryCrossEntropyBackward0>) tensor(0.6425)
11 tensor(0.8197, grad_fn=<BinaryCrossEntropyBackward0>) tensor(0.6425)
12 tensor(1.0957, grad_fn=<BinaryCrossEntropyBackward0>) tensor(0.6166)
13 tensor(0.9089, grad_fn=<BinaryCrossEntropyBackward0>) tensor(0.6344)
14

11. Compute the accuracy and cross entropy loss for the trained logistic regression model for the test data.

In [68]:

for data, label in test_dataloader:
    #TODO: Call the model on the training data
    p = model(data)
    loss = criterion(p.squeeze(),label*1.0)
    acc = accuracy(p.squeeze(), label)
    print(loss, acc)

tensor(0.4957, grad_fn=<BinaryCrossEntropyBackward0>) tensor(0.8012)


12. Using PyTorch, perform Maximum Likelihood Estimation (MLE) for a non-linear Bernoulli
prediction model for the binary classification dataset.

In [69]:
class nn(torch.nn.Module):
    def __init__(self, inputSize, hiddenSize, outputSize):
        super(nn, self).__init__()
        #TODO: Call a linear layer constructor
        self.layer1 = torch.nn.Linear(inputSize,hiddenSize)
        #TODO: Call a linear layer constructor
        self.layer2 = torch.nn.Linear(hiddenSize,hiddenSize)
        #TODO: Call a linear layer constructor
        self.layer3 = torch.nn.Linear(hiddenSize,outputSize)

    def forward(self, x):
        x = torch.flatten(x, start_dim=1)
        #TODO: Call layer 1 on x using an ReLU activation function
        h1 = torch.nn.functional.relu(self.layer1(x))
        #TODO: Call layer 2 on h1 using an ReLU activation function
        h2 = torch.nn.functional.relu(self.layer2(h1))
        #TODO: Call layer 3 on h2 using a sigmoid activation function
        return torch.sigmoid(self.layer3(h2))


In [70]:
model = nn(tensor_size, 200, 1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(n_epochs):
    for data, label in train_dataloader:
        optimizer.zero_grad()
        #TODO: Call the model on the training data
        p = model(data)
        loss =  criterion(p.squeeze(),label*1.0)
        acc = accuracy(p.squeeze(), label)
        loss.backward()
        optimizer.step()
        if verbose_option: print(epoch, loss, acc)

0 tensor(0.6917, grad_fn=<BinaryCrossEntropyBackward0>) tensor(0.4469)
1 tensor(0.6140, grad_fn=<BinaryCrossEntropyBackward0>) tensor(0.6981)
2 tensor(0.5630, grad_fn=<BinaryCrossEntropyBackward0>) tensor(0.7606)
3 tensor(0.5433, grad_fn=<BinaryCrossEntropyBackward0>) tensor(0.7713)
4 tensor(0.5118, grad_fn=<BinaryCrossEntropyBackward0>) tensor(0.7869)
5 tensor(0.4789, grad_fn=<BinaryCrossEntropyBackward0>) tensor(0.8066)
6 tensor(0.4580, grad_fn=<BinaryCrossEntropyBackward0>) tensor(0.8253)
7 tensor(0.4426, grad_fn=<BinaryCrossEntropyBackward0>) tensor(0.8334)
8 tensor(0.4243, grad_fn=<BinaryCrossEntropyBackward0>) tensor(0.8409)
9 tensor(0.4045, grad_fn=<BinaryCrossEntropyBackward0>) tensor(0.8506)
10 tensor(0.3880, grad_fn=<BinaryCrossEntropyBackward0>) tensor(0.8562)
11 tensor(0.3748, grad_fn=<BinaryCrossEntropyBackward0>) tensor(0.8591)
12 tensor(0.3620, grad_fn=<BinaryCrossEntropyBackward0>) tensor(0.8669)
13 tensor(0.3478, grad_fn=<BinaryCrossEntropyBackward0>) tensor(0.8747)
14

13. Compute the accuracy and cross entropy loss for the trained non-linear classification model for the test data.

In [71]:
for data, label in test_dataloader:
    #TODO: Call the model on the training data
    p = model(data)
    loss =  criterion(p.squeeze(),label*1.0)
    acc = accuracy(p.squeeze(), label)
    print(loss, acc)

tensor(0.0927, grad_fn=<BinaryCrossEntropyBackward0>) tensor(0.9787)
